<a href="https://colab.research.google.com/github/varadbarclays/dev1/blob/main/ICAIF_DocRanker_Varad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ACM ICAIF-25 AI Agentic Retrieval Grand Challenge

---

## Overview

The competition consists of two main ranking tasks:

1. **Document Ranking** – Identify and rank the five most relevant documents.  
2. **Chunk Ranking** – Identify and rank the five most relevant text chunks.

---

## Processing Pipeline

- **Data Loading**  
  Reads evaluation data from JSONL files.

- **Token Analysis**  
  Checks input size to decide whether it exceed the context length of the model..

- **Smart Ranking**  
  - *Normal cases*: Single-stage ranking.  
  - *High-token cases*: Multi-stage divide-and-conquer ranking to handle large inputs efficiently.

- **Result Compilation**  
  Combines model outputs and prepares a final CSV submission file.

---

## Output

- **kaggle_submission.csv**  
  Ready-to-submit file containing the required `sample_id` and `target_index` columns.

- **Comprehensive Statistics**  
  Summarized metrics and analysis of ranking results.

- **Top-5 Rankings**  
  Returns the five most relevant items for each query as required by the challenge.

---

## Usage

This notebook enables **end-to-end evaluation**: from loading data to generating a Kaggle-ready submission file.  
Simply run the pipeline and upload the generated `kaggle_submission.csv` to the competition platform.

In [ ]:
!pip install openai tiktoken python-dotenv pydantic tqdm

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
import re

import asyncio
import csv
import json
import os
import traceback
from typing import Dict, List

import tiktoken
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel
from tqdm.asyncio import tqdm

import torch
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, losses
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments
from datasets import Dataset
from transformers import TrainingArguments
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

load_dotenv()

False

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"varadsrivastava","key":"07be8d5799432154520a4aff77f26a0d"}'}

In [ ]:
!ls -lha kaggle.json
!pip install -q kaggle # installing the kaggle package
!mkdir -p ~/.kaggle # creating .kaggle folder where the key should be placed
!cp kaggle.json ~/.kaggle/ # move the key to the folder
!pwd # checking the present working directory

-rw-r--r-- 1 root root 71 Oct 12 11:40 kaggle.json
/content


In [ ]:
# giving rw access (if 401-nathorized)

# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# import os
# os.environ["TORCH_ALLOW_BF16_CUDA"] = "0"

# Dataset

In [ ]:
!kaggle competitions download -c acm-icaif-25-ai-agentic-retrieval-grand-challenge

 99% 1.22G/1.23G [00:08<00:00, 246MB/s]
100% 1.23G/1.23G [00:08<00:00, 151MB/s]


In [ ]:
!unzip *acm-icaif-25-ai-agentic-retrieval-grand-challenge.zip

Archive:  acm-icaif-25-ai-agentic-retrieval-grand-challenge.zip
  inflating: chunk_ranking_kaggle_dev.jsonl  
  inflating: chunk_ranking_kaggle_eval.jsonl  
  inflating: document_ranking_kaggle_dev.jsonl  
  inflating: document_ranking_kaggle_eval.jsonl  
  inflating: kaggle_submission.csv   


# Documents

In [ ]:
file_path = '/content/document_ranking_kaggle_dev.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_doc_dev = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_doc_dev.head())

Head of /content/document_ranking_kaggle_dev.jsonl:


,uuid,messages,qrel
0,qe100cdf8e8f5,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}"
1,q723817294fbe,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '0': 2, '1': 1, '3': 1, '2': 0}"
2,qd79970afaa57,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '1': 2, '0': 1, '2': 1, '3': 0}"
3,qeb144d0e9991,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 3, '2': 2, '4': 1, '0': 0, '3': 0}"
4,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '2': 2, '1': 1, '0': 0, '3': 0}"


In [ ]:
df_doc_dev

,uuid,messages,qrel
0,qe100cdf8e8f5,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 1, '0': 0, '1': 0, '2': 0, '3': 0}"
1,q723817294fbe,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '0': 2, '1': 1, '3': 1, '2': 0}"
2,qd79970afaa57,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '1': 2, '0': 1, '2': 1, '3': 0}"
3,qeb144d0e9991,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 3, '2': 2, '4': 1, '0': 0, '3': 0}"
4,q4f1cc6ea4ba6,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 3, '2': 2, '1': 1, '0': 0, '3': 0}"
...,...,...,...
4981,q30f7c8056c93,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 4, '2': 3, '0': 2, '4': 1, '3': 0}"
4982,q402c59dfe9b1,"[{'role': 'user', 'content': 'Rank the followi...","{'1': 4, '0': 3, '4': 2, '3': 1, '2': 0}"
4983,q45a9a878195e,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '2': 3, '1': 2, '0': 1, '3': 0}"
4984,q2c57b56ff808,"[{'role': 'user', 'content': 'Rank the followi...","{'4': 4, '1': 3, '2': 2, '0': 1, '3': 0}"


In [ ]:
df_doc_dev["messages"][0]

[{'role': 'user',
  'content': 'Rank the following financial document types by relevance to answer the question. Provide your ranking as a list of indices from most relevant to least relevant.\n\nQuestion: How has Agilent Technologies’ instrument reliability metric for its core diagnostics manufacturing process changed recently?\n\nDocument Types to rank:\n[Document Index 0] DEF14A\n\n[Document Index 1] 10-K\n\n[Document Index 2] 10-Q\n\n[Document Index 3] 8-K\n\n[Document Index 4] Earnings\n\nYour response must be a list of indices in exact list format (e.g., [4, 2, 1, 0, 3]), ranking every index from 0 to 4 by most relevant document type index to least relevant document type index.'}]

In [ ]:
file_path = '/content/document_ranking_kaggle_eval.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_doc_eval = pd.DataFrame(data)
print(f"Head of {file_path}:")
display(df_doc_eval.head())

Head of /content/document_ranking_kaggle_eval.jsonl:


,_id,messages
0,doc_q39d7b7,"[{'role': 'user', 'content': 'Rank the followi..."
1,doc_q8edbb8,"[{'role': 'user', 'content': 'Rank the followi..."
2,doc_q060a50,"[{'role': 'user', 'content': 'Rank the followi..."
3,doc_q3ec868,"[{'role': 'user', 'content': 'Rank the followi..."
4,doc_qc7db20,"[{'role': 'user', 'content': 'Rank the followi..."


In [ ]:
# extract the question from df_doc_dev between "Question: " to "\n\nDocument"
df_doc_dev["question"] = df_doc_dev["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])

df_doc_eval["question"] = df_doc_eval["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])


In [ ]:
df_doc_dev["question"][2]

'How do sustainability or ESG considerations influence customer demand in Agilent Technologies’ market?'

In [ ]:
df_doc_eval["question"][0]

'How has Salesforce’s subscription and support segment profitability trended over recent periods?'

# Training Reranker on sample with ListNet custom loss

## 1. Load Cross-Encoder Model

In [ ]:
model = CrossEncoder(
    # "google/embeddinggemma-300m",
    # "FinLang/finance-embeddings-investopedia",
    "sentence-transformers/all-mpnet-base-v2",
    # "cross-encoder/ms-marco-MiniLM-L12-v2",
    #sentence-transformers/all-mpnet-base-v2",#colbert-ir/colbertv2.0", #microsoft/mpnet-base","jinaai/jina-reranker-v2-base-multilingual"
    num_labels=1,  # Single output for ranking score
    # activation_fn=torch.nn.Sigmoid(),
    trust_remote_code=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
# model = CrossEncoder(
#     "ProsusAI/finbert",
#     num_labels=1,   # force single output
#     trust_remote_code=True,
#     device='cuda' if torch.cuda.is_available() else 'cpu',
#     automodel_args={"ignore_mismatched_sizes": True}  # allow head reset
# )

In [ ]:
print(model)

CrossEncoder(
  (model): MPNetForSequenceClassification(
    (mpnet): MPNetModel(
      (embeddings): MPNetEmbeddings(
        (word_embeddings): Embedding(30527, 768, padding_idx=1)
        (position_embeddings): Embedding(514, 768, padding_idx=1)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): MPNetEncoder(
        (layer): ModuleList(
          (0-11): 12 x MPNetLayer(
            (attention): MPNetAttention(
              (attn): MPNetSelfAttention(
                (q): Linear(in_features=768, out_features=768, bias=True)
                (k): Linear(in_features=768, out_features=768, bias=True)
                (v): Linear(in_features=768, out_features=768, bias=True)
                (o): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affi

## 2. Load the document ranking dev set


In [ ]:
docno_to_name = {
    "0": "[DOC=DEF-14A | proxy statement | governance, compensation, shareholder voting matters | annual filing | meeting details, proxy rules, dissenters' rights, solicitation info, interests in decisions, security changes, voting procedures | composition of the board of directors and how they oversee the management of the company | executive compensation practices and philosophy, tables of executive and director compensation components | tables of major stockholder ownership percentages]",
    "1": "[DOC=10-K | annual report | company's history, organizational structure, financial statements, earnings per share, subsidiaries | business - company’s main operations, products and services | risk factors - risks the company faces or may face in the future | financial statements, income statement, balance sheets, cash flows | MD&A, management’s discussion and analysis of financial condition and results of operations, business results from the previous fiscal year]",
    "2": "[DOC=10-Q | quarterly report | condensed financial statements, management discussion, analysis of the financial condition, disclosures regarding market risk, and internal controls | legal proceedings, unregistered sales of equity securities, the use of proceeds from the sale of unregistered sales of equity securities, and defaults upon senior securities]",
    "3": "[DOC=8-K | current report | material events, timely disclosures | material agreements, bankruptcy filings, and mine safety violations | financial information: acquisition or disposition of assets, material impairments, and changes in shell company status | securities and trading markets: delistings, failures to meet listing standards, and unregistered sales of equity securities | accountants and financial statements: changes in auditors and non-reliance on previously issued financial statements | corporate governance and management: changes in control, director departures, executive officer appointments, and amendments to governing documents]",
    "4": "[DOC=Earnings | earnings call transcript | forward guidance, Q&A, management commentary | quarterly | talk | recent | questions]"
}

In [ ]:
for i in range(5):
  print(len(docno_to_name[str(i)]))

484
468
356
650
128


In [ ]:
file_path = '/content/document_ranking_kaggle_dev.jsonl'
data = []
with open(file_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))
df_doc_dev = pd.DataFrame(data)

# Filter rows where qrel values include 0, 1, 2, 3, and 4
required_values = set([0, 1, 2, 3, 4])
df_doc_dev = df_doc_dev[df_doc_dev['qrel'].apply(lambda x: required_values.issubset(set(x.values())))]

# Extract the question from df_doc_dev
df_doc_dev["question"] = df_doc_dev["messages"].apply(lambda x: x[0]["content"].split("Question: ")[1].split("\n\nDocument")[0])


# 2. Prepare Training Data
def create_training_data(df_doc_dev):
    """
    Create training dataset with queries and document type rankings
    Format: Each query has all 5 document types with relevance scores [4,3,2,1,0]
    """

    training_examples = []
    name_to_docno = {"DEF14A": "0", "10-K": "1", "10-Q": "2", "8-K": "3", "Earnings": "4"}
    # docno_to_name = {"0": "DEF14A", "1": "10-K", "2": "10-Q", "3": "8-K", "4": "Earnings"}

#     docno_to_name = {
#     "0": "[DOC=DEF-14A | proxy statement | governance, compensation, shareholder voting matters | annual filing]",
#     "1": "[DOC=10-K | annual report | comprehensive business overview, risks, financials | 100-300 pages]",
#     "2": "[DOC=10-Q | quarterly report | interim financials, MD&A updates | 30-60 pages]",
#     "3": "[DOC=8-K | current report | material events, timely disclosures | ad-hoc filing]",
#     "4": "[DOC=Earnings | earnings call transcript | forward guidance, Q&A, management commentary | quarterly]"
# }


    for index, row in df_doc_dev.iterrows():
        query = row['question']
        qrel = row['qrel']
        messages = row['messages'][0]['content']

        docs = [docno_to_name[x] for x in qrel.keys()]
        labels = list(qrel.values())

        training_examples.append({
            "query": query,
            "docs": docs,
            "labels": labels
        })

    return training_examples

## 3. Format Data for ListNet Loss


In [ ]:
def format_data_for_listnet(training_examples):
    """
    Convert training examples to format required by ListNet loss
    """
    formatted_data = {
        "query": [],
        "docs": [],
        "labels": []
    }

    for example in training_examples:
        formatted_data["query"].append(example["query"])
        formatted_data["docs"].append(example["docs"])
        formatted_data["labels"].append(example["labels"])

    return formatted_data

# Create Dataset
training_examples = create_training_data(df_doc_dev)
formatted_data = format_data_for_listnet(training_examples)

## 4. Train-test split

In [ ]:
formatted_data["docs"][0]

["[DOC=10-K | annual report | company's history, organizational structure, financial statements, earnings per share, subsidiaries | business - company’s main operations, products and services | risk factors - risks the company faces or may face in the future | financial statements, income statement, balance sheets, cash flows | MD&A, management’s discussion and analysis of financial condition and results of operations, business results from the previous fiscal year]",
 '[DOC=10-Q | quarterly report | condensed financial statements, management discussion, analysis of the financial condition, disclosures regarding market risk, and internal controls | legal proceedings, unregistered sales of equity securities, the use of proceeds from the sale of unregistered sales of equity securities, and defaults upon senior securities]',
 '[DOC=Earnings | earnings call transcript | forward guidance, Q&A, management commentary | quarterly | talk | recent | questions]',
 "[DOC=DEF-14A | proxy statement 

In [ ]:
# Split into train/test
train_data, test_data = train_test_split(
    list(zip(formatted_data["query"], formatted_data["docs"], formatted_data["labels"])),
    test_size=0.2,
    random_state=42
)

# Split into train/validation
train_data, val_data = train_test_split(train_data,
    test_size=0.2,
    random_state=42
)

In [ ]:
train_data[0]

('What did Fifth Third Bancorp’s leadership say about Fifth Third Bancorp’s dividend policy?',
 ["[DOC=10-K | annual report | company's history, organizational structure, financial statements, earnings per share, subsidiaries | business - company’s main operations, products and services | risk factors - risks the company faces or may face in the future | financial statements, income statement, balance sheets, cash flows | MD&A, management’s discussion and analysis of financial condition and results of operations, business results from the previous fiscal year]",
  '[DOC=10-Q | quarterly report | condensed financial statements, management discussion, analysis of the financial condition, disclosures regarding market risk, and internal controls | legal proceedings, unregistered sales of equity securities, the use of proceeds from the sale of unregistered sales of equity securities, and defaults upon senior securities]',
  "[DOC=DEF-14A | proxy statement | governance, compensation, shareho

In [ ]:
train_data[5]

('How does GE HealthCare Technologies view the pace of innovation cycles and their effect on market competitiveness?',
 ['[DOC=Earnings | earnings call transcript | forward guidance, Q&A, management commentary | quarterly | talk | recent | questions]',
  "[DOC=10-K | annual report | company's history, organizational structure, financial statements, earnings per share, subsidiaries | business - company’s main operations, products and services | risk factors - risks the company faces or may face in the future | financial statements, income statement, balance sheets, cash flows | MD&A, management’s discussion and analysis of financial condition and results of operations, business results from the previous fiscal year]",
  "[DOC=DEF-14A | proxy statement | governance, compensation, shareholder voting matters | annual filing | meeting details, proxy rules, dissenters' rights, solicitation info, interests in decisions, security changes, voting procedures | composition of the board of directo

In [ ]:
# Create datasets
train_dataset = Dataset.from_dict({
    "query": [item[0] for item in train_data],
    "docs": [item[1] for item in train_data],
    "labels": [item[2] for item in train_data]
})

val_dataset = Dataset.from_dict({
    "query": [item[0] for item in val_data],
    "docs": [item[1] for item in val_data],
    "labels": [item[2] for item in val_data]
})

test_dataset = Dataset.from_dict({
    "query": [item[0] for item in test_data],
    "docs": [item[1] for item in test_data],
    "labels": [item[2] for item in test_data]
})

## 5. Initialize ListNet Loss


In [ ]:
# loss_function = losses.ListNetLoss(
#     model=model,
#     activation_fn=torch.nn.Identity(),  # No activation since we want raw scores
#     mini_batch_size=None  # Process full batches
# )

In [ ]:
loss_function = losses.LambdaLoss(
    model=model,
    activation_fn=torch.nn.Identity(),  # No activation since we want raw scores
    mini_batch_size=None  # Process full batches
)

## 6. Training Arguments

In [ ]:
# training_args = CrossEncoderTrainingArguments(
#     output_dir="./MARCOMINILM",
#     num_train_epochs=6,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     warmup_steps=50,
#     learning_rate=2e-5,
#     logging_dir="./logs",
#     logging_steps=50,
#     eval_strategy="steps",
#     eval_steps=50,
#     save_steps=400,
#     save_total_limit=3,
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better=False,
#     gradient_accumulation_steps=2,
#     dataloader_drop_last=False,
#     report_to="none",

#     # 🚨 force FP16, disable BF16
#     fp16=False,
#     bf16=True,

#     # 🚨 required in some PyTorch/HF versions to stop bf16 fallback
#     optim="adamw_torch"
# )


from sentence_transformers.cross_encoder import CrossEncoderTrainingArguments

training_args = CrossEncoderTrainingArguments(
    output_dir="./mpnet-v2-financial-doc-ranker",

    # Training configuration
    learning_rate=2e-5,
    per_device_train_batch_size=4,  # Small batch size for ListNet
    gradient_accumulation_steps=4,  # Effective batch size = 16
    num_train_epochs=5,

    # Scheduler and warmup
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,

    # Evaluation strategy
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=8,

    # Saving strategy
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=True,

    # Memory optimization
    fp16=True,
    # gradient_checkpointing=True,
    dataloader_num_workers=2,

    # Logging
    logging_steps=25,
    report_to=["none"],

    # Reproducibility
    seed=42,
    data_seed=42
)



## 7. Initialize Trainer


In [ ]:
def ndcg_at_k(y_true, y_pred, k=5):
    """Compute nDCG@k for a single query."""
    order = np.argsort(y_pred)[::-1]
    y_true = np.take(y_true, order)

    dcg = dcg_at_k(y_true, k)
    ideal_dcg = dcg_at_k(sorted(y_true, reverse=True), k)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0

In [ ]:
trainer = CrossEncoderTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss_function,
    # compute_metrics=ndcg_at_k,
)

# 8. Train the Model
print("Starting training...")
trainer.train()

Starting training...


Step,Training Loss,Validation Loss
50,0.720700,0.657566
100,0.570700,0.532729
150,0.521000,0.507364
200,0.532700,0.511346
250,0.499100,0.492231
300,0.475700,0.475871
350,0.471700,0.481403
400,0.496600,0.470267
450,0.453200,0.476464
500,0.467200,0.468165


TrainOutput(global_step=790, training_loss=0.4938601276542567, metrics={'train_runtime': 1225.974, 'train_samples_per_second': 10.29, 'train_steps_per_second': 0.644, 'total_flos': 0.0, 'train_loss': 0.4938601276542567, 'epoch': 5.0})

## 8. Save the model

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=True)

In [ ]:
model.save("./jina")
model.push_to_hub("Pranjal2002/all-mpnet-base-v3")

print("Model saved successfully!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...4e9jd0r/model.safetensors:   1%|1         | 6.50MB /  438MB            

Model saved successfully!


## 9. Eval on Test Set

In [ ]:
import numpy as np
from tqdm import tqdm

# ---------- Helper Metric Functions ----------
def dcg_at_k(relevances, k):
    """Compute DCG@k given a list of relevances (sorted by predicted rank)."""
    relevances = np.asarray(relevances, dtype=float)[:k]
    if relevances.size:
        return np.sum((2**relevances - 1) / np.log2(np.arange(2, relevances.size + 2)))
    return 0.0

def ndcg_at_k(y_true, y_pred, k=5):
    """Compute nDCG@k for a single query."""
    order = np.argsort(y_pred)[::-1]
    y_true = np.take(y_true, order)

    dcg = dcg_at_k(y_true, k)
    ideal_dcg = dcg_at_k(sorted(y_true, reverse=True), k)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0

def average_precision_at_k(y_true, y_pred, k=5):
    """Compute AP@k for a single query."""
    order = np.argsort(y_pred)[::-1]
    y_true = np.take(y_true, order)

    hits = 0
    sum_precisions = 0.0
    for i in range(min(k, len(y_true))):
        if y_true[i] > 0:  # relevant doc
            hits += 1
            sum_precisions += hits / (i + 1.0)
    return sum_precisions / hits if hits > 0 else 0.0

def reciprocal_rank_at_k(y_true, y_pred, k=5):
    """Compute MRR@k for a single query."""
    order = np.argsort(y_pred)[::-1]
    y_true = np.take(y_true, order)

    for i in range(min(k, len(y_true))):
        if y_true[i] > 0:
            return 1.0 / (i + 1.0)
    return 0.0

# ---------- Evaluation Loop ----------
def evaluate_ranking(model, val_data, k=5):
    metrics = {"ndcg": [], "map": [], "mrr": []}

    for query, docs, labels in tqdm(val_data, desc="Evaluating"):
        # CrossEncoder expects list of [query, doc] pairs
        pairs = [[query, doc] for doc in docs]
        scores = model.predict(pairs)

        # Compute metrics for this query
        metrics["ndcg"].append(ndcg_at_k(labels, scores, k))
        metrics["map"].append(average_precision_at_k(labels, scores, k))
        metrics["mrr"].append(reciprocal_rank_at_k(labels, scores, k))

    # Average over queries
    results = {m: float(np.mean(v)) for m, v in metrics.items()}
    return results

results = evaluate_ranking(model, test_data, k=5)
print("Test Metrics:", results)
# results = evaluate_ranking(model, val_data, k=5)
# print("Validation Metrics:", results)

Evaluating: 100%|██████████| 789/789 [00:40<00:00, 19.37it/s]

Test Metrics: {'ndcg': 0.9022714969388541, 'map': 0.9493081960287283, 'mrr': 0.9847908745247148}


## Results (NDCG@5, MAP@5, MRR@5)
1. ColBert
- Validation Metrics: {'ndcg': 0.8981391762909982, 'map': 0.9380708716235031, 'mrr': 0.9847535505430243}
- test Metrics: {'ndcg': 0.9009670751662266, 'map': 0.9355739256290357, 'mrr': 0.9834168336673347}
2. Jina /jina-reranker-v2-base-multilingual
-  test Metrics: {'ndcg': 0.8861880085968683, 'map': 0.9227524493431306, 'mrr': 0.969188376753507}
- Validation Metrics: {'ndcg': 0.882754863371895, 'map': 0.9216966026176553, 'mrr': 0.9710735171261486}

3. FINBERT
- Test Metrics: {'ndcg': 0.9045412893519893, 'map': 0.9382570696949455, 'mrr': 0.9816633266533067}
- Validation Metrics: {'ndcg': 0.9005019647498531, 'map': 0.9400306321358952, 'mrr': 0.9843358395989975}

4. cross-encoder/ms-marco-MiniLM-L12-v2
- Test Metrics: {'ndcg': 0.9056648690056422, 'map': 0.949397972116603, 'mrr': 0.9860583016476553}
- Metrics: {'ndcg': 0.9029903152397294, 'map': 0.951472530375066, 'mrr': 0.9849445324881141}

5. colbert_new
- Test Metrics: {'ndcg': 0.9017529866976771, 'map': 0.9498521335023237, 'mrr': 0.9879594423320659}
- Metrics: {'ndcg': 0.8991222025489771, 'map': 0.9501584786053882, 'mrr': 0.9873217115689382}

6. finbert_new
- Test Metrics: {'ndcg': 0.9006964656047587, 'map': 0.9501584283903675, 'mrr': 0.9873257287705957}
- Metrics: {'ndcg': 0.9002681952386852, 'map': 0.95141970417327, 'mrr': 0.9896988906497622}

7. mpnetv2-new
- Test Metrics: {'ndcg': 0.9065571838413917, 'map': 0.9508924799324038, 'mrr': 0.9873257287705957}





## Figure out the Best model/approach and train it on the whole dataset

In [ ]:
train_dataset

Dataset({
    features: ['query', 'docs', 'labels'],
    num_rows: 2523
})

In [ ]:
val_dataset

Dataset({
    features: ['query', 'docs', 'labels'],
    num_rows: 631
})

In [ ]:
test_dataset

Dataset({
    features: ['query', 'docs', 'labels'],
    num_rows: 789
})

In [ ]:
from datasets import concatenate_datasets

# join train, val and test
full_dataset = concatenate_datasets([train_dataset, val_dataset, test_dataset])
full_dataset

Dataset({
    features: ['query', 'docs', 'labels'],
    num_rows: 3943
})

In [ ]:
training_args = CrossEncoderTrainingArguments(
    output_dir="./mpnet-v2-financial-doc-ranker",

    # Training configuration
    learning_rate=2e-5,
    per_device_train_batch_size=4,  # Small batch size for ListNet
    gradient_accumulation_steps=4,  # Effective batch size = 16
    num_train_epochs=5,

    # Scheduler and warmup
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,

    # Evaluation strategy
    eval_strategy="no",
    eval_steps=50,
    per_device_eval_batch_size=8,

    # Saving strategy
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    # load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=True,

    # Memory optimization
    fp16=True,
    # gradient_checkpointing=True,
    dataloader_num_workers=2,

    # Logging
    logging_steps=25,
    report_to=["none"],

    # Reproducibility
    seed=42,
    data_seed=42
)

In [ ]:
trainer = CrossEncoderTrainer(
    model=model,
    args=training_args,
    train_dataset=full_dataset,
    # eval_dataset=val_dataset,
    loss=loss_function,
    # compute_metrics=compute_metrics,

)

# 8. Train the Model
print("Starting training...")
trainer.train()

Starting training...


Step,Training Loss
25,1.608500
50,1.594200
75,1.484800
100,1.405000
125,1.405900
150,1.363500
175,1.353500
200,1.347200
225,1.336800
250,1.329100


TrainOutput(global_step=1235, training_loss=1.3150423482362075, metrics={'train_runtime': 1552.3116, 'train_samples_per_second': 12.7, 'train_steps_per_second': 0.796, 'total_flos': 0.0, 'train_loss': 1.3150423482362075, 'epoch': 5.0})

## Now Evaluate the Best model on Eval set and save the outputs

In [ ]:
# 10. Evaluate final model on eval set
def evaluate_model(model, test_query):
    """
    Evaluate the trained model on a test query
    """

    doc_types = list(docno_to_name.values())

    # Create query-document pairs
    pairs = [(test_query, doc_type) for doc_type in doc_types]

    # Get scores
    scores = model.predict(pairs)

    # Create ranking
    doc_scores = list(zip(doc_types, scores))
    doc_scores.sort(key=lambda x: x[1], reverse=True)

    print(f"\nQuery: {test_query}")
    print("Document Type Rankings:")
    for i, (doc_type, score) in enumerate(doc_scores, 1):
        print(f"{i}. {doc_type}: {score:.4f}")

    return doc_scores

# 11. Test the Model
if __name__ == "__main__":
    # Load the fine-tuned model for testing
    # trained_model = CrossEncoder("/content/jina/checkpoint-1995")
    trained_model = model

    # Test queries
    # test_queries = [
    #     "What are the company's financial performance metrics?",
    #     "Are there any recent merger announcements?",
    #     "What is discussed in shareholder meetings?",
    #     "What were the latest quarterly results?"
    # ]

    rankings = []
    for query in df_doc_eval["question"]:
        ranking = evaluate_model(trained_model, query)
        rankings.append(ranking)
        print("-" * 50)


Query: How has Salesforce’s subscription and support segment profitability trended over recent periods?
Document Type Rankings:
1. [DOC=10-K | annual report | comprehensive business overview, risks, financials | 100-300 pages]: 0.7701
2. [DOC=10-Q | quarterly report | interim financials, MD&A updates | 30-60 pages]: 0.7581
3. [DOC=Earnings | earnings call transcript | forward guidance, Q&A, management commentary | quarterly]: 0.4727
4. [DOC=8-K | current report | material events, timely disclosures | ad-hoc filing]: 0.3924
5. [DOC=DEF-14A | proxy statement | governance, compensation, shareholder voting matters | annual filing]: 0.1645
--------------------------------------------------

Query: How does Blackstone manage equity award burn rate or share pool availability?
Document Type Rankings:
1. [DOC=10-K | annual report | comprehensive business overview, risks, financials | 100-300 pages]: 0.8170
2. [DOC=DEF-14A | proxy statement | governance, compensation, shareholder voting matters

In [ ]:
rankings

[[('[DOC=10-K | annual report | comprehensive business overview, risks, financials | 100-300 pages]',
   np.float32(0.77007014)),
  ('[DOC=10-Q | quarterly report | interim financials, MD&A updates | 30-60 pages]',
   np.float32(0.7580507)),
  ('[DOC=Earnings | earnings call transcript | forward guidance, Q&A, management commentary | quarterly]',
   np.float32(0.47267002)),
  ('[DOC=8-K | current report | material events, timely disclosures | ad-hoc filing]',
   np.float32(0.3923858)),
  ('[DOC=DEF-14A | proxy statement | governance, compensation, shareholder voting matters | annual filing]',
   np.float32(0.16446745))],
 [('[DOC=10-K | annual report | comprehensive business overview, risks, financials | 100-300 pages]',
   np.float32(0.81697804)),
  ('[DOC=DEF-14A | proxy statement | governance, compensation, shareholder voting matters | annual filing]',
   np.float32(0.7931336)),
  ('[DOC=10-Q | quarterly report | interim financials, MD&A updates | 30-60 pages]',
   np.float32(0.5708

In [ ]:
# Create a reversed dictionary to map document names back to keys
name_to_docno = {v: k for k, v in docno_to_name.items()}

# Convert rankings to lists of keys
ranked_keys = []
for ranking in rankings:
    current_ranked_keys = []
    for doc_name, score in ranking:
        # Find the corresponding key for the document name
        key = name_to_docno.get(doc_name)
        if key is not None:
            current_ranked_keys.append(int(key)) # Convert key to integer
    ranked_keys.append(current_ranked_keys)

print("Ranked Keys:")
display(ranked_keys)

Ranked Keys:


[[1, 2, 4, 3, 0],
 [1, 0, 2, 3, 4],
 [4, 0, 3, 1, 2],
 [2, 1, 3, 4, 0],
 [4, 0, 3, 1, 2],
 [1, 0, 4, 2, 3],
 [1, 2, 3, 4, 0],
 [1, 0, 2, 3, 4],
 [4, 0, 3, 1, 2],
 [1, 2, 4, 0, 3],
 [1, 2, 4, 3, 0],
 [4, 0, 3, 1, 2],
 [4, 1, 2, 0, 3],
 [4, 1, 2, 0, 3],
 [1, 0, 2, 3, 4],
 [4, 1, 2, 0, 3],
 [2, 1, 3, 4, 0],
 [4, 1, 0, 2, 3],
 [1, 4, 2, 0, 3],
 [1, 2, 4, 3, 0],
 [4, 1, 0, 2, 3],
 [1, 4, 2, 0, 3],
 [1, 2, 4, 0, 3],
 [4, 1, 2, 0, 3],
 [1, 2, 4, 0, 3],
 [4, 1, 2, 0, 3],
 [1, 2, 4, 3, 0],
 [1, 4, 0, 2, 3],
 [0, 1, 4, 2, 3],
 [1, 4, 0, 2, 3],
 [4, 1, 2, 3, 0],
 [1, 4, 2, 3, 0],
 [4, 1, 0, 2, 3],
 [4, 1, 2, 3, 0],
 [4, 1, 2, 0, 3],
 [4, 0, 3, 1, 2],
 [1, 2, 3, 4, 0],
 [1, 2, 4, 0, 3],
 [1, 2, 3, 4, 0],
 [1, 2, 4, 3, 0],
 [4, 1, 0, 2, 3],
 [1, 0, 2, 4, 3],
 [4, 1, 2, 0, 3],
 [4, 2, 1, 3, 0],
 [1, 4, 2, 0, 3],
 [1, 2, 4, 0, 3],
 [1, 2, 4, 0, 3],
 [1, 2, 0, 4, 3],
 [0, 1, 2, 3, 4],
 [4, 1, 2, 3, 0],
 [4, 1, 2, 0, 3],
 [1, 2, 4, 3, 0],
 [1, 2, 4, 3, 0],
 [1, 2, 4, 3, 0],
 [2, 1, 3, 4, 0],
 [1, 2, 0,

In [ ]:
df_doc_eval["rankings"] = ranked_keys

In [ ]:
#create empty df with columns
results = pd.DataFrame(columns=['sample_id', 'target_index'])

for index, i in enumerate(df_doc_eval['_id']):
  for j in range(0,5):
    current_id = i
    current_rankings = df_doc_eval['rankings'][index][j]

    # for key, value in current_rankings.items():
    #   if value == j:
    doc_rank = {'sample_id': current_id, 'target_index': current_rankings}
    # add doc rank to df
    results = pd.concat([results, pd.DataFrame([doc_rank])], ignore_index=True)

In [ ]:
results

,sample_id,target_index
0,doc_q39d7b7,1
1,doc_q39d7b7,2
2,doc_q39d7b7,4
3,doc_q39d7b7,3
4,doc_q39d7b7,0
...,...,...
995,doc_qc6264f,1
996,doc_qc6264f,4
997,doc_qc6264f,2
998,doc_qc6264f,0


In [ ]:
df_doc_eval['rankings'][0]

[1, 2, 4, 3, 0]

In [ ]:
# Save the submission file
submission_file_path = 'submission_docs_mpnet.csv'
results.to_csv(submission_file_path, index=False)

print(f"Submission file created successfully at: {submission_file_path}")

Submission file created successfully at: submission_docs_mpnet.csv


In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=True)

In [ ]:

model.push_to_hub("varadsrivastava/findocranker-mpnet-base-v2")

print("Model saved successfully!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...q3xdmji/model.safetensors:   0%|          | 22.2kB /  438MB            

Model saved successfully!


In [ ]:
df_doc_eval

,_id,messages,question
0,doc_q39d7b7,"[{'role': 'user', 'content': 'Rank the followi...",How has Salesforce’s subscription and support ...
1,doc_q8edbb8,"[{'role': 'user', 'content': 'Rank the followi...",How does Blackstone manage equity award burn r...
2,doc_q060a50,"[{'role': 'user', 'content': 'Rank the followi...",What questions were asked about A. O. Smith Co...
3,doc_q3ec868,"[{'role': 'user', 'content': 'Rank the followi...","How has the ratio of Corpay, Inc.’s recurring ..."
4,doc_qc7db20,"[{'role': 'user', 'content': 'Rank the followi...",What questions were asked about Ford Motor Com...
...,...,...,...
195,doc_qd70b93,"[{'role': 'user', 'content': 'Rank the followi...",How do investors view Ameren Corporation’s val...
196,doc_q21c207,"[{'role': 'user', 'content': 'Rank the followi...",What dependency risks exist for Cincinnati Fin...
197,doc_qd29700,"[{'role': 'user', 'content': 'Rank the followi...",How has the ratio of Applied Materials’ recurr...
198,doc_qc0665c,"[{'role': 'user', 'content': 'Rank the followi...","What dependency risks exist for Assurant, Inc...."
